# Study 945 — The Hidden Financing 💳

**What interest rate are you really paying inside a leveraged ETF?**

Buy $10,000 of a 2x S&P fund and you control $20,000 of index. Somebody lent you the other
$10,000. The fact sheet quotes an expense ratio; it does not quote the *interest rate* on
that loan, which arrives silently inside a swap spread. But it is recoverable, because the
fund's arithmetic is rigid:

$$r_{fund} = L \cdot r_{index} - (L-1)\cdot \frac{f}{252} - \frac{ER}{252} + \varepsilon$$

Regress the fund's daily total return on the benchmark's: the **slope** is the realised
leverage, the **intercept** is the whole daily drag. Annualise it, strip the published
expense ratio, divide by the `L−1` dollars actually borrowed — what is left is *f*, the
implied financing rate.

We do it on **SSO** (2x) and **UPRO** (3x) against **SPY** total return, 2009-06-26 →
2026-06-30 (4,276 days), and race the answer against **^IRX** (the 13-week
T-bill) and against what a margin desk would charge.

*Numbers below are the frozen headline (`docs/results.md`, Fingerprint `1c447f91bfae`); the
live cells run the offline synthetic control. As-of 2026-06-30.*


## 1. The loan nobody quotes you

A leveraged ETF is two things bolted together: an index position and a margin loan. You are told the price of the wrapper (the expense ratio). You are not told the price of the loan. So we read it off the tape instead — the fund's return has to be the index return times the leverage, *minus* the fee, *minus* the interest. Everything except the interest is known, so the interest falls out.

In [1]:
R = {'start': '2009-06-26', 'end': '2026-06-30', 'n_days': 4276, 'fp': '1c447f91bfae', 'irx_mean': 1.396, 'bil_realised': 1.267, 'sso_beta': 1.9972, 'sso_beta_t': -0.27, 'sso_r2': 0.99748, 'sso_alpha': -2.774, 'sso_alpha_se': 0.208, 'sso_alpha_t': -13.31, 'sso_drag': 2.962, 'sso_f': 2.072, 'sso_spread': 0.677, 'sso_allin': 2.962, 'sso_allin_spread': 1.567, 'upro_beta': 2.9894, 'upro_beta_t': -0.63, 'upro_r2': 0.99653, 'upro_alpha': -4.787, 'upro_alpha_se': 0.375, 'upro_alpha_t': -12.75, 'upro_drag': 5.07, 'upro_f': 2.08, 'upro_spread': 0.684, 'upro_allin': 2.535, 'upro_allin_spread': 1.139, 'boot_sso': 0.721, 'boot_sso_lo': 0.39, 'boot_sso_hi': 1.079, 'boot_upro': 0.768, 'boot_upro_lo': 0.432, 'boot_upro_hi': 1.114, 'gspc_sso_drag': -0.888, 'gspc_sso_f': -1.778, 'gspc_upro_drag': -0.694, 'gspc_upro_f': -0.802, 'roll_sso_mean': 0.545, 'roll_sso_sd': 0.437, 'roll_sso_slope': 1.058, 'roll_sso_int': 0.466, 'roll_sso_corr': 0.975, 'roll_upro_mean': 0.603, 'roll_upro_sd': 0.405, 'roll_upro_slope': 1.061, 'roll_upro_int': 0.52, 'roll_upro_corr': 0.979, 'sso_spread_se': 0.208, 'sso_spread_t': 3.25, 'sso_allin_t': 7.52, 'upro_spread_se': 0.188, 'upro_spread_t': 3.65, 'upro_allin_t': 6.07, 'irx_bey': 1.42, 'era_e_spread': 0.374, 'era_l_spread': 0.951, 'era_e_t': 1.74, 'era_l_t': 3.33, 'zirp_spread': 0.658, 'hiked_spread': 0.635, 'zirp_t': 2.69, 'hiked_t': 2.08, 'upro_era_e_spread': 0.38, 'upro_era_l_spread': 0.962, 'upro_era_e_t': 1.77, 'upro_era_l_t': 3.89, 'upro_zirp_spread': 0.631, 'upro_hiked_spread': 0.711, 'upro_zirp_t': 2.83, 'upro_hiked_t': 2.8, 'be_sso_early': 1.167, 'be_sso_late': 1.607, 'be_sso_zirp': 1.169, 'be_sso_hiked': 1.748, 'be_upro_early': 0.811, 'be_upro_late': 1.255, 'be_upro_zirp': 0.827, 'be_upro_hiked': 1.364, 'er_lo_spread': 0.817, 'er_hi_spread': 0.517, 'sharpe_spy': 0.836, 'sharpe_sso': 0.792, 'sharpe_upro': 0.792, 'race_sso_075': -0.64, 'race_sso_075_t': -4.56, 'race_sso_150': 0.11, 'race_sso_150_t': 0.78, 'race_sso_400': 2.61, 'race_sso_400_t': 18.59, 'race_sso_600': 4.61, 'race_sso_600_t': 32.84, 'race_upro_075': -0.57, 'race_upro_075_t': -2.25, 'race_upro_400': 5.93, 'race_upro_400_t': 23.36, 'race_upro_600': 9.93, 'race_upro_600_t': 39.12, 'be_sso_gross': 1.427, 'be_sso': 1.39, 'be_sso_5bp': 1.245, 'be_upro_gross': 1.09, 'be_upro': 1.035, 'be_upro_5bp': 0.817, 'turn_sso': 1.443, 'turn_upro': 4.33, 'turn_cost_sso': 0.04, 'turn_cost_upro': 0.11, 'syn_planted': 0.75, 'syn_2x': 0.683, 'syn_3x': 0.813, 'syn_err_2x': -0.067, 'syn_err_3x': 0.063}
print('Mean 13-week T-bill rate over the window : %.3f%%' % R['irx_mean'])
print()
rows = (('SSO  (2x)', R['sso_f'], R['sso_spread'], R['sso_allin'], R['sso_allin_spread']),
        ('UPRO (3x)', R['upro_f'], R['upro_spread'], R['upro_allin'], R['upro_allin_spread']))
for tag, f, sp, allin, allsp in rows:
    print('%s implied borrowing rate %5.3f%%  = T-bills %+.3f pp' % (tag, f, sp))
    print('%s all-in, incl. the fee   %5.3f%%  = T-bills %+.3f pp'
          % (' ' * 9, allin, allsp))

Mean 13-week T-bill rate over the window : 1.396%

SSO  (2x) implied borrowing rate 2.072%  = T-bills +0.677 pp
          all-in, incl. the fee   2.962%  = T-bills +1.567 pp
UPRO (3x) implied borrowing rate 2.080%  = T-bills +0.684 pp
          all-in, incl. the fee   2.535%  = T-bills +1.139 pp


## 2. The two funds agree to within one basis point

SSO borrows one dollar per dollar of your money; UPRO borrows two. Solve each one separately and they imply the *same* borrowing rate: **2.072%** and **2.080%** — eight tenths of a basis point apart. Against a mean T-bill rate of 1.396%, that is a mark-up of about **+0.68 pp** for borrowing money, with a HAC *t* of **3.25** and **3.65**.

Two cautions, because they matter more than the agreement does:

- The two funds are **both ProShares**, sharing an issuer and a swap desk. Agreeing tells you the arithmetic is not broken; it is not two independent witnesses to the level.
- What is left after stripping the fee is the interest **plus** every other friction inside the wrapper — swap spreads, the cost of the daily reset, tracking loss. A return regression cannot separate them, so read 2.07% as an **upper bound** on the interest rate, not the rate.

> 🔬 *For the quants:* the intercepts are −2.774%/yr (*t* = -13.31) and −4.787%/yr (*t* = -12.75) — but those *t*'s only say the drag is non-zero, which the fee alone guarantees. The claim is the spread, at *t* = 3.25 / 3.65. Realised betas 1.9972 and 2.9894, neither distinguishable from its stated leverage, so the intercept is a level term and not a bad slope.

## 3. The 3x fund is the *cheaper* loan

Counter-intuitive, and it falls straight out of where the fee lands. The expense ratio is charged on your **whole** stake; the loan is only part of it. At 2x you pay the fee on $1 to borrow $1. At 3x you pay (almost) the same fee on $1 to borrow **$2** — so the fee is spread over twice the borrowing.

| | borrowed per $1 | drag on your money | all-in cost per borrowed dollar |
|---|--:|--:|--:|
| **SSO** (2x) | $1.00 | 2.96%/yr | **2.96%** (T-bills +1.57) |
| **UPRO** (3x) | $2.00 | 5.07%/yr | **2.54%** (T-bills +1.14) |

None of which says 3x is *safer* — it is far more violent, and this desk has said so twice already (studies 61 and 100). It says only that the interest you pay per borrowed dollar is lower.

## 4. When the Fed moves, your loan moves — one for one

Re-estimate the rate on a rolling one-year window through 2009-2026 and the implied borrowing rate tracks the T-bill rate almost perfectly (correlation **0.975**), with a slope of **1.06** and a roughly constant mark-up on top. It was 0.11% in 2013 and 6.06% in 2024 — the whole rate cycle, passed through.

So the wrapper is not a fixed-rate loan you locked in. It is a floating-rate loan, and whatever the Fed does to short rates lands on you within the year.

## 5. So — wrapper, or your own broker?

That is the whole practical question, and the tape answers it with a single number: the **break-even margin rate**. Hold 2x SPY yourself on margin, reset daily exactly as the fund does, pay one basis point each time you trade. You tie the fund when your broker charges **T-bills + 1.39%**; at 3x the line is **T-bills + 1.03%**.

| Your broker charges | 2x: wrapper minus DIY | 3x: wrapper minus DIY |
|---|--:|--:|
| bills + 0.75% (prime-broker tier) | **-0.64%/yr** | **-0.57%/yr** |
| bills + 1.50% (low-cost retail) | +0.11%/yr | — |
| bills + 4.00% (mainstream broker) | **+2.61%/yr** | **+5.93%/yr** |
| bills + 6.00% (full-service retail) | **+4.61%/yr** | **+9.93%/yr** |

Almost every ordinary brokerage account is on the bottom two rows, where the wrapper wins by several percent a year. If you are on the top row you are already an institution and you knew that.

Two honest limits on that table. **You may not be allowed to do it yourself:** Reg T caps a US retail margin account at 2x to begin with, so the whole 3x column is a price comparison rather than a choice you can make. And the break-even is a full-sample average that moves when you cut the sample — ^IRX +1.17% to +1.61% at 2x across the two halves. It never climbs anywhere near the 4-6% a retail desk charges, which is why the answer holds; but it is a range, not a constant.

> 🔬 *For the quants:* the broker rates are **labelled assumptions** from public rate cards; the break-even itself contains no broker assumption at all. Both arms are raced excess-of-cash against BIL's realised total return, with one execution lag and the daily reset's turnover charged one-way against NAV. No margin call is modelled — that flatters the DIY arm, so the break-even is a conservative line for the wrapper.

## 6. Live check — does the arithmetic actually recover a known rate? (offline)

We build synthetic wrappers whose financing rate we *chose*: benchmark + 75 bp, plus a 0.90% fee, over a 24-year tape with a rate cycle in it. Then we run the exact estimator used above and see whether it hands the 75 bp back. And we run it again on a null world where the wrappers borrow at exactly the benchmark rate and charge nothing — where it must return zero.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from lev_financing import data, strategy as st
print('SYNTHETIC (offline) — not the real tape')
for ss, tag in ((1.0, 'planted 75 bp'), (0.0, 'null      0 bp')):
    for L in (2, 3):
        got = [st.synthetic_detect(*data.synthetic_panel(signal_strength=ss, seed=945+s),
                                   leverage=L)['spread_over_rate_pct'] for s in range(6)]
        print('  %s  %dx -> recovered spread %+.3f pp (6 seeds)' % (tag, L, np.mean(got)))

SYNTHETIC (offline) — not the real tape


  planted 75 bp  2x -> recovered spread +0.683 pp (6 seeds)


  planted 75 bp  3x -> recovered spread +0.813 pp (6 seeds)


  null      0 bp  2x -> recovered spread -0.067 pp (6 seeds)


  null      0 bp  3x -> recovered spread +0.063 pp (6 seeds)


## Verdict

- **Signal — Real.** The charge is measured, not guessed. The mark-up over T-bills is **+0.68 pp** at a HAC *t* of **3.25** (SSO) and **3.65** (UPRO), and the all-in charge — which needs no guess about the fee at all — is **+1.57 pp** at *t* = **7.5**. Robust to any plausible error in the assumed expense ratio, and positive in both rate worlds.
- **Three caveats that ride with the badge.** It is an **upper bound** on interest (swap and reset frictions are in there too). The **first half of the sample is weak** — +0.37 pp at *t* = 1.74, below the bar — so what is solid is the post-2018 level near +0.95. And these are the funds that **survived**; the ones that closed are not in the average, so this is a floor for the class.
- **Tradability — Investable.** Not an edge — a cost decision, and a big one. All-in you pay bills + 1.57 pp at 2x and +1.14 pp at 3x. Against a mainstream retail margin desk that is worth **+2.6% to +9.9% a year** in your favour — a gap wide enough that the wobble in the break-even (^IRX +1.17% to +1.61% across the two halves) never threatens the answer. Against a prime broker it is worth **-0.64%** against you — but that tier, and the 3x do-it-yourself arm Reg T does not allow a retail account, are not on offer to the reader this rule is written for. **If you are levering 2x anyway, the wrapper is the cheaper loan unless your broker lends under about bills + 1.2%.**
- **What this does not say.** Nothing here is an argument for *using* leverage. It prices the loan; the shape of the ride — the volatility drag, the −60% and −77% drawdowns these two funds actually took — is studies 61, 100 and 944.